# Classification Partielle Multi-Objectif avec JMetalPy
## OAFD – M2 MIAGE

Ce notebook implémente la classification partielle **multi-objectif** (représentation Michigan).  
Au lieu de maximiser la F1-mesure (un seul objectif), on optimise **simultanément** :
- **Objectif 1** : Confiance (Precision) à maximiser  
- **Objectif 2** : Sensibilité (Recall) à maximiser  

**Algorithmes testés :** NSGA-II et SPEA2  
**Jeux de données :** Yeast1 et PIMA Diabetes  
**Comparaison avec :** Random Forest, SVM, C4.5 (Scikit-Learn)

## 1. Imports et chargement des données

In [ ]:
# =============================================================================
# CELLULE 1 – IMPORTS ET DÉPENDANCES
# =============================================================================
# Ce bloc charge toutes les bibliothèques utilisées dans le projet.
# Il doit être exécuté EN PREMIER avant toute autre cellule.

# ── Librairies standard Python ───────────────────────────────────────────────
import random          # Générateur de nombres pseudo-aléatoires (reproductibilité via seed)
import warnings        # Contrôle des messages d'avertissement Python
import time            # Mesure du temps d'exécution (benchmarking des algorithmes)
import numpy as np     # Calcul numérique vectorisé : tableaux, stats, opérations mathématiques
import pandas as pd    # Lecture et manipulation de fichiers CSV en DataFrames
import matplotlib.pyplot as plt        # Bibliothèque de visualisation (graphiques 2D)
import matplotlib.patches as mpatches  # Création de formes/légendes personnalisées

warnings.filterwarnings('ignore')  # Désactive les warnings non critiques (JMetalPy, sklearn)
%matplotlib inline                 # Force l'affichage des figures dans le notebook Jupyter

# ── Scikit-Learn – Machine Learning classique ────────────────────────────────
from sklearn.model_selection import StratifiedKFold, train_test_split
# StratifiedKFold : validation croisée k-fold qui préserve la proportion de classes positives
# train_test_split : partage aléatoire stratifié des données en train (70%) / test (30%)

from sklearn.ensemble import RandomForestClassifier  # Forêt aléatoire : combinaison d'arbres (boîte noire)
from sklearn.svm import SVC                          # Machine à Vecteurs de Support (boîte noire)
from sklearn.tree import DecisionTreeClassifier      # Arbre de décision de profondeur limitée (boîte blanche)

from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score
# precision_score : TP / (TP + FP) → parmi les prédictions positives, combien sont vraies ?
# recall_score    : TP / (TP + FN) → parmi tous les positifs réels, combien sont détectés ?
# f1_score        : 2 * (Precision * Recall) / (Precision + Recall) → compromis P/R
# accuracy_score  : (TP + TN) / Total → taux de bonnes classifications globales

# ── JMetalPy – Noyau du cadre d'optimisation MO ─────────────────────────────
from jmetal.core.problem import Problem          # Classe abstraite à hériter pour définir un problème MO
from jmetal.core.solution import FloatSolution   # Représentation d'une solution à variables réelles

from jmetal.operator.mutation import PolynomialMutation
# Mutation polynomiale : modifie chaque variable avec probabilité 1/n_vars
# Le paramètre distribution_index contrôle l'amplitude de la perturbation (grand = petite perturbation)

from jmetal.operator.crossover import SBXCrossover
# SBX (Simulated Binary Crossover) : croisement inspiré du croisement binaire à un point
# appliqué aux variables continues. distribution_index contrôle la dispersion des enfants.

from jmetal.util.termination_criterion import StoppingByEvaluations
# Critère d'arrêt basé sur le nombre d'évaluations de la fonction objectif (budget fixe)

from jmetal.util.solution import get_non_dominated_solutions
# Extrait le sous-ensemble des solutions non-dominées (front de Pareto approximé)

# ── JMetalPy – Algorithmes évolutionnaires MO ────────────────────────────────
from jmetal.algorithm.multiobjective.nsgaii import NSGAII
# NSGA-II : algorithme MO de référence (Deb et al., 2002)
# Utilise un tri par non-dominance et une distance de crowding pour maintenir la diversité

from jmetal.algorithm.multiobjective.spea2 import SPEA2
# SPEA2 : algorithme MO avec archive externe (Zitzler et al., 2001)
# Calcule une "force" de dominance et une densité d'individus pour le tri

# ── JMetalPy – Indicateur de qualité de front ────────────────────────────────
from jmetal.core.quality_indicator import HyperVolume
# Hypervolume (HV) : volume de l'espace objectif dominé par le front de Pareto
# Mesure la qualité globale d'un front (convergence + diversité)
# Calculé par rapport à un point de référence (ici [0.05, 0.05])

print('✓ Tous les imports OK')


In [ ]:
# =============================================================================
# CELLULE 2 – CHARGEMENT DES DONNÉES
# =============================================================================
# Lit les deux jeux de données depuis des fichiers CSV locaux et prépare
# les matrices de features (X) et les vecteurs de labels (y) pour chaque dataset.

# ── Dataset 1 : PIMA Indians Diabetes ───────────────────────────────────────
df_pima = pd.read_csv('pima_diabetes.csv')          # Lecture du fichier CSV en DataFrame
X_pima  = df_pima.drop('Outcome', axis=1).to_numpy()  # Retire la colonne cible → matrice X (768×8)
y_pima  = df_pima['Outcome'].to_numpy()               # Extrait la colonne cible → vecteur y (0=sain, 1=diabétique)
features_pima = list(df_pima.columns[:-1])            # Liste des 8 noms d'attributs (Glucose, BMI, etc.)

# ── Dataset 2 : Yeast1 (protéines) ──────────────────────────────────────────
df_yeast = pd.read_csv('yeast1.csv')                   # Lecture du fichier CSV Yeast1
X_yeast  = df_yeast.drop('Output', axis=1).to_numpy()  # Matrice X (1484×8 attributs biochimiques)
y_yeast  = df_yeast['Output'].to_numpy()                # Vecteur y (0=autre localisation, 1=cytoplasme)
features_yeast = list(df_yeast.columns[:-1])            # Liste des 8 noms d'attributs

# ── Affichage des statistiques de base ──────────────────────────────────────
# shape[0]=nb d'instances, sum()=nb de positifs, (y==0).sum()=nb de négatifs
print(f'PIMA   : {X_pima.shape}  | Positifs: {y_pima.sum()}  | Négatifs: {(y_pima==0).sum()}')
print(f'Yeast1 : {X_yeast.shape} | Positifs: {y_yeast.sum()} | Négatifs: {(y_yeast==0).sum()}')
print(f'\nAttributs PIMA   : {features_pima}')
print(f'Attributs Yeast1 : {features_yeast}')


## 2. Rappel des résultats Scikit-Learn (Étape 1)

Nous réexécutons les trois algorithmes de référence avec le **même protocole** :  
validation croisée 3-fold stratifiée, seed fixe = 42.  
On collecte **confiance** (precision) et **sensibilité** (recall) pour la comparaison avec les fronts Pareto.

In [ ]:
# =============================================================================
# CELLULE 3 – BASELINES SCIKIT-LEARN (VALIDATION CROISÉE 3-FOLD)
# =============================================================================
# Évalue trois classifieurs classiques en validation croisée stratifiée.
# Ces résultats servent de référence (baseline) pour la comparaison finale
# avec les algorithmes MO (NSGA-II et SPEA2).

def run_sklearn(X, y, model_fn, n_folds=3, seed=42):
    """
    Évalue un modèle Scikit-Learn en validation croisée stratifiée.

    Paramètres :
        X        : matrice des features (numpy array)
        y        : vecteur des labels (0/1)
        model_fn : fonction lambda prenant une seed et renvoyant un modèle sklearn
        n_folds  : nombre de plis (défaut=3)
        seed     : graine pour StratifiedKFold (non utilisée ici, la seed est dans model_fn)

    Retourne : dict avec precision, recall, f1 (moyennes et écarts-types sur les k plis)
    """
    skf = StratifiedKFold(n_splits=n_folds)   # Découpe les données en 3 plis équilibrés
    all_prec, all_rec, all_f1 = [], [], []     # Listes pour accumuler les scores par pli

    for train_idx, test_idx in skf.split(X, y):
        Xtr, Xte = X[train_idx], X[test_idx]  # Données d'entraînement et de test du pli
        ytr, yte = y[train_idx], y[test_idx]  # Labels correspondants
        m = model_fn(seed)                    # Instancie le modèle avec la seed fixée
        m.fit(Xtr, ytr)                       # Entraînement sur le pli d'entraînement
        yp = m.predict(Xte)                   # Prédiction sur le pli de test
        all_prec.append(precision_score(yte, yp, zero_division=0))  # Précision du pli
        all_rec.append(recall_score(yte, yp, zero_division=0))      # Rappel du pli
        all_f1.append(f1_score(yte, yp, zero_division=0))           # F1 du pli

    # Retourne les moyennes et écarts-types sur les k plis
    return {
        'precision': np.mean(all_prec), 'recall': np.mean(all_rec), 'f1': np.mean(all_f1),
        'prec_std': np.std(all_prec),   'rec_std': np.std(all_rec),  'f1_std': np.std(all_f1)
    }

# ── Définition des modèles baseline sous forme de fonctions lambda ────────────
# lambda s : permet de créer une nouvelle instance avec une seed donnée à la demande
rf_fn  = lambda s: RandomForestClassifier(n_estimators=100, random_state=s)  # 100 arbres
svm_fn = lambda s: SVC(kernel='rbf', random_state=s)                         # Noyau gaussien RBF
c45_fn = lambda s: DecisionTreeClassifier(max_depth=4, random_state=s)       # Arbre limité à 4 niveaux

# ── Évaluation de tous les modèles sur les deux datasets ─────────────────────
sklearn_results = {}   # Dict de résultats : sklearn_results[dataset][algo] = métriques

for ds_name, X, y in [('PIMA', X_pima, y_pima), ('Yeast1', X_yeast, y_yeast)]:
    sklearn_results[ds_name] = {}
    for algo_name, fn in [('RF', rf_fn), ('SVM', svm_fn), ('C4.5', c45_fn)]:
        r = run_sklearn(X, y, fn)                               # Évaluation par CV
        sklearn_results[ds_name][algo_name] = r                # Stockage des résultats
        # Affichage formaté : F1±std | Prec±std | Rec±std
        print(f'[{ds_name}] {algo_name:5s} | F1={r["f1"]:.3f}±{r["f1_std"]:.3f}'
              f' | Prec={r["precision"]:.3f}±{r["prec_std"]:.3f}'
              f' | Rec={r["recall"]:.3f}±{r["rec_std"]:.3f}')
    print()  # Ligne vide entre les deux datasets

print('✓ Baselines Scikit-Learn calculées')


## 3. Définition du Problème Multi-Objectif JMetalPy

### Représentation Michigan
Une solution = un vecteur de `2 × n_attributs` flottants.  
Chaque paire `(lower_i, upper_i)` définit les bornes d'un attribut :
- `lower_i > upper_i` → attribut **désactivé**  
- `lower_i ≤ upper_i` → attribut **activé**

### Deux objectifs (minimisation JMetalPy)
| Objectif | Formule | Sens |
|----------|---------|------|
| obj[0] | −Confiance (−Precision) | minimiser ↔ maximiser Precision |
| obj[1] | −Sensibilité (−Recall) | minimiser ↔ maximiser Recall |

Le **point de référence** pour l'Hypervolume est `[0.05, 0.05]`  
(légèrement au-dessus de 0 pour inclure les solutions avec score = 1.0).

In [ ]:
# =============================================================================
# CELLULE 4 – CLASSE DU PROBLÈME MO + FONCTIONS UTILITAIRES
# =============================================================================
# Définit le problème de classification partielle multi-objectif (Michigan)
# et les fonctions d'aide utilisées dans tout le notebook.

class PartialClassifMO(Problem):
    """
    Classification Partielle – Problème multi-objectif (représentation Michigan).

    Chaque solution encode UNE SEULE RÈGLE de la forme :
        SI attr_0 ∈ [lo_0, hi_0] ET attr_1 ∈ [lo_1, hi_1] ET ... → classe positive

    Variables :  2 * n_attributes flottants (paires [lo_i, hi_i] par attribut)
    Objectifs :  -Precision  et  -Recall  (négation pour minimisation)

    Un attribut est "activé" si lo_i <= hi_i.
    Si lo_i > hi_i, l'attribut est ignoré dans la règle (ne filtre rien).
    """

    def __init__(self, X: np.ndarray, y: np.ndarray):
        super().__init__()  # Appel du constructeur de la classe Problem JMetalPy
        self.X = X                              # Matrice des exemples d'entraînement
        self.y = y                              # Vecteur des étiquettes (0/1)
        self.n_attributes = X.shape[1]          # Nombre d'attributs (colonnes de X)
        self.pos_indices = np.where(y == 1)[0]  # Indices des exemples positifs (diabétiques, etc.)
        self.neg_indices = np.where(y == 0)[0]  # Indices des exemples négatifs

        # ── Construction des bornes de l'espace de recherche ─────────────────
        self.lower_bound, self.upper_bound = [], []
        for i in range(self.n_attributes):
            cmin = float(np.min(X[:, i]))  # Valeur minimale observée pour l'attribut i
            cmax = float(np.max(X[:, i]))  # Valeur maximale observée pour l'attribut i
            if cmin == cmax:               # Cas dégénéré : attribut constant → légère perturbation
                cmin -= 0.001; cmax += 0.001
            # Pour chaque attribut : 2 bornes [lo_i, hi_i]
            # La borne supérieure est étendue de +1 pour permettre la désactivation (lo > hi)
            self.lower_bound += [cmin, cmin]
            self.upper_bound += [cmax + 1.0, cmax + 1.0]

        self.obj_directions = [self.MINIMIZE, self.MINIMIZE]  # Les deux objectifs sont à minimiser
        self.obj_labels = ['-Precision', '-Recall']           # Labels pour les graphiques

    # ── Méthodes obligatoires de l'interface JMetalPy ────────────────────────
    def number_of_variables(self):   return 2 * self.n_attributes  # 2 vars par attribut
    def number_of_objectives(self):  return 2                       # -Precision, -Recall
    def number_of_constraints(self): return 0                       # Pas de contraintes

    def _decode(self, solution):
        """
        Décode une solution en vecteur de prédictions y_pred.
        Pour chaque exemple j, prédit 1 si l'exemple vérifie tous les intervalles activés.
        """
        # Conversion en float (nécessaire car JMetalPy peut renvoyer des nombres complexes)
        vars_ = [float(v.real) if isinstance(v, complex) else float(v)
                 for v in solution.variables]

        # Reconstruction des paires (lo_i, hi_i) pour chaque attribut
        bornes = [(vars_[i], vars_[i+1]) for i in range(0, len(vars_), 2)]

        # Garde seulement les attributs activés (lo <= hi)
        activated = [(i, lo, hi) for i, (lo, hi) in enumerate(bornes) if lo <= hi]

        # Si aucun attribut n'est activé, la règle est vide → prédit 0 partout
        if not activated:
            return np.zeros(len(self.y), dtype=int)

        # Pour chaque exemple j, vérifie si tous les attributs activés sont dans leur intervalle
        return np.array([
            1 if all(lo <= self.X[j, i] <= hi for i, lo, hi in activated) else 0
            for j in range(len(self.X))
        ])

    def evaluate(self, solution: FloatSolution) -> FloatSolution:
        """
        Calcule les deux objectifs (-Precision, -Recall) pour une solution donnée.
        Les valeurs sont stockées dans solution.objectives[0] et [1].
        """
        y_pred = self._decode(solution)  # Prédictions selon la règle encodée
        if y_pred.sum() == 0:
            # Si la règle ne couvre aucun exemple → Precision et Recall = 0
            # On stocke 0.0 (équivalent à -Precision=0 et -Recall=0 après négation)
            solution.objectives[0] = 0.0
            solution.objectives[1] = 0.0
        else:
            # Objectifs = valeurs NÉGATIVES (minimiser -f = maximiser f)
            solution.objectives[0] = -float(precision_score(self.y, y_pred, zero_division=0))
            solution.objectives[1] = -float(recall_score(self.y, y_pred, zero_division=0))
        return solution

    def create_solution(self) -> FloatSolution:
        """
        Génère une solution initiale (individu) aléatoire.
        Stratégie : sélectionne 1 exemple positif + 1 négatif, active 2 attributs aléatoires
        avec un intervalle couvrant les deux exemples sur ces attributs.
        """
        sol = FloatSolution(self.lower_bound, self.upper_bound,
                            self.number_of_objectives(), self.number_of_constraints())

        ind_pos = self.X[np.random.choice(self.pos_indices)]  # Exemple positif aléatoire
        ind_neg = self.X[np.random.choice(self.neg_indices)]  # Exemple négatif aléatoire

        # Choisit 2 attributs à activer (les autres seront désactivés)
        active = set(np.random.choice(self.n_attributes, size=2, replace=False))

        vars_ = []
        for i in range(self.n_attributes):
            vp, vn = float(ind_pos[i]), float(ind_neg[i])
            if i in active:
                # Intervalle = [min(vp, vn), max(vp, vn)] → couvre les deux exemples
                vars_ += [min(vp, vn), max(vp, vn)]
            else:
                # Désactivé : lo > hi (la règle ignore cet attribut)
                vars_ += [vp + 1.0, vp]
        sol.variables = vars_
        return sol

    def name(self): return 'PartialClassifMO'


# =============================================================================
# FONCTIONS UTILITAIRES
# =============================================================================

def predict(solution_vars, X):
    """
    Calcule le vecteur y_pred pour un ensemble X à partir des variables d'une solution.
    Utilisé pour l'évaluation sur le jeu de test (X_te différent de X_train).
    """
    vars_ = [float(v.real) if isinstance(v, complex) else float(v) for v in solution_vars]
    bornes = [(vars_[i], vars_[i+1]) for i in range(0, len(vars_), 2)]
    activated = [(i, lo, hi) for i, (lo, hi) in enumerate(bornes) if lo <= hi]
    if not activated:
        return np.zeros(len(X), dtype=int)
    return np.array([
        1 if all(lo <= X[j, i] <= hi for i, lo, hi in activated) else 0
        for j in range(len(X))
    ])

def decode_rule(solution_vars, feature_names):
    """
    Traduit les variables d'une solution en règle lisible par un humain.
    Ex : 'Règle : SI Glucose ∈ [120.000, 180.000] ET BMI ∈ [30.000, 45.000] → classe positive'
    """
    bornes = [(solution_vars[i], solution_vars[i+1]) for i in range(0, len(solution_vars), 2)]
    parts = [
        f'{feature_names[i]} ∈ [{lo:.3f}, {hi:.3f}]'
        for i, (lo, hi) in enumerate(bornes) if lo <= hi   # Seulement les attributs activés
    ]
    rule = 'SI ' + ' ET '.join(parts) + ' → classe positive' if parts else 'Règle vide'
    print('Règle :', rule)
    return rule

def compute_hv(solutions, ref_point=None):
    """
    Calcule l'hypervolume du front de Pareto d'une liste de solutions.
    ref_point : point de référence dans l'espace des objectifs (minimisation).
                Doit dominer toutes les solutions du front.
                Par défaut [0.05, 0.05] → correspond à -Precision=0.05, -Recall=0.05.
    """
    if not solutions:
        return 0.0  # Front vide → HV nul
    front = np.array([s.objectives for s in solutions])  # Matrice (n_solutions × 2)
    if ref_point is None:
        ref_point = [0.05, 0.05]   # Point de référence par défaut
    hv_calc = HyperVolume(ref_point)
    return hv_calc.compute(front)  # Calcul de l'HV (volume dominé dans l'espace 2D)

def get_pareto_front(solutions):
    """
    Extrait les solutions non-dominées (front de Pareto) depuis une population.
    Une solution A domine B si elle est au moins aussi bonne sur tous les objectifs
    et strictement meilleure sur au moins un.
    """
    return get_non_dominated_solutions(solutions)

print('✓ Classe PartialClassifMO et fonctions utilitaires définies')


## 4. Paramètres expérimentaux

### 4.1 Paramètres fixes

Les paramètres suivants sont **identiques** pour les deux algorithmes et toutes les configurations :

| Paramètre | Valeur | Justification |
|-----------|--------|---------------|
| Prob. croisement | 0.9 | Valeur standard recommandée pour SBX |
| Index croisement | 20 | Distribution concentrée autour des parents |
| Prob. mutation | 1/n_vars | Règle du pouce : une variable mutée par individu |
| Index mutation | 20 | Perturbations modérées |
| Max évaluations | **déterminé section 4.3** | Fixé automatiquement par étude de convergence |
| Validation croisée | — | Pas utilisée (train/test split stratifié 70/30) |
| N_RUNS | 20 | Minimum requis pour significativité statistique |

### 4.2 Choix des tailles de population

Trois tailles sont testées pour isoler l'effet de ce seul paramètre :

| Taille | Motivation |
|--------|-----------|
| **20** | Population petite – convergence rapide mais diversité faible |
| **50** | Taille intermédiaire – bon équilibre exploration/exploitation |
| **100** | Grande population – meilleure couverture du front Pareto, coût accru |

Ces tailles couvrent un ordre de grandeur (×5) permettant d'observer l'impact sur la qualité du front Pareto.

In [ ]:
# =============================================================================
# CELLULE 5 – PARAMÈTRES EXPÉRIMENTAUX ET SPLIT TRAIN/TEST
# =============================================================================
# Définit tous les hyperparamètres fixes utilisés dans les expériences
# et effectue le découpage des données en ensembles d'entraînement et de test.

# ── Paramètres des opérateurs génétiques (NE PAS MODIFIER) ───────────────────
CROSSOVER_PROB  = 0.9    # Probabilité d'appliquer le croisement SBX à une paire de parents
CROSSOVER_INDEX = 20.0   # distribution_index SBX : grand = enfants proches des parents
MUTATION_INDEX  = 20.0   # distribution_index mutation polynomiale : grand = petite perturbation
MAX_EVALUATIONS = 500    # Budget d'évaluations (défini définitivement après l'étude de convergence)
N_RUNS          = 20     # Nombre d'exécutions indépendantes par configuration (robustesse statistique)
REF_POINT       = [0.05, 0.05]  # Point de référence pour l'Hypervolume
# REF_POINT = [0.05, 0.05] signifie que l'on mesure le volume pour des solutions
# avec -Precision < 0.05 ET -Recall < 0.05, i.e. Precision > 0.95 ET Recall > 0.95

# ── Tailles de population à comparer ─────────────────────────────────────────
POP_SIZES = [20, 50, 100]  # 3 configurations de population testées par algorithme/dataset

# ── Split train/test stratifié 70/30 ─────────────────────────────────────────
# test_size=0.3 : 30% des données pour le test, 70% pour l'entraînement
# stratify=y    : conserve la même proportion de positifs dans les deux partitions
# random_state=42 : graine fixe → résultats reproductibles
from sklearn.model_selection import train_test_split

X_pima_tr,  X_pima_te,  y_pima_tr,  y_pima_te  = train_test_split(
    X_pima, y_pima, test_size=0.3, stratify=y_pima, random_state=42)

X_yeast_tr, X_yeast_te, y_yeast_tr, y_yeast_te = train_test_split(
    X_yeast, y_yeast, test_size=0.3, stratify=y_yeast, random_state=42)

# ── Récapitulatif de la configuration ────────────────────────────────────────
print(f'PIMA   train={X_pima_tr.shape}  test={X_pima_te.shape}')
print(f'Yeast1 train={X_yeast_tr.shape} test={X_yeast_te.shape}')
print(f'\nConfiguration expérimentale :')
print(f'  Algorithmes     : NSGA-II, SPEA2')
print(f'  Tailles de pop  : {POP_SIZES}')
print(f'  Runs par config : {N_RUNS}')
print(f'  Max évaluations : {MAX_EVALUATIONS}')
# Calcul du nombre total de runs : 3 pop × 20 runs × 2 algos × 2 datasets = 240 runs
print(f'  Total de runs   : {len(POP_SIZES) * N_RUNS * 2 * 2} (3 pop × 20 runs × 2 algos × 2 datasets)')


## 4.3 Étude de convergence – Détermination du nombre d'évaluations optimal

Avant de lancer les 20 runs, on détermine le **nombre d'évaluations nécessaire** pour que les algorithmes convergent.

**Protocole :**
- 1 seul run (seed = 42) par budget testé
- Population fixée à **50** (taille intermédiaire)
- Budgets testés : 100, 250, 500, 1000, 2500, 5000, 7500, 10 000
- Métrique : **Hypervolume** du front Pareto obtenu
- Les 2 algorithmes (NSGA-II et SPEA2) sont testés sur les 2 datasets

👉 On choisit le budget à partir duquel la courbe **se stabilise (plateau)** : gain marginal < 1%.

In [ ]:
# =============================================================================
# CELLULE 6 – ÉTUDE DE CONVERGENCE (CALIBRAGE DU BUDGET D'ÉVALUATIONS)
# =============================================================================
# Détermine empiriquement le budget MAX_EVALUATIONS optimal pour NSGA-II sur PIMA.
# Principe : tester plusieurs budgets et tracer l'hypervolume en fonction du budget.
# Le plateau de la courbe indique le budget à partir duquel l'amélioration est négligeable.

# ── Budgets à tester (progression quasi-logarithmique) ───────────────────────
EVAL_BUDGETS = [100, 250, 500, 1000, 2500, 5000, 7500, 10_000]
CONV_POP     = 50    # Taille de population fixe pour cette étude (compromis vitesse/qualité)
CONV_SEED    = 42    # Graine fixe pour reproductibilité de l'étude de convergence

# Seul NSGA-II sur PIMA est testé (représentatif) pour limiter le temps d'exécution
CONV_ALGO    = 'NSGA-II'
CONV_DATASET = 'PIMA'


def run_convergence_study(X_train, y_train, algo_name, budgets=EVAL_BUDGETS,
                           pop_size=CONV_POP, seed=CONV_SEED):
    """
    Lance 1 run par budget et retourne la liste des hypervolumes correspondants.
    Permet de tracer la courbe HV = f(budget) pour identifier le plateau de convergence.
    """
    hvs = []  # Liste des hypervolumes obtenus pour chaque budget
    for budget in budgets:
        # Fixation des graines aléatoires pour reproductibilité
        random.seed(seed); np.random.seed(seed)

        # Instanciation du problème pour chaque budget (état frais)
        problem   = PartialClassifMO(X_train, y_train)
        n_vars    = 2 * X_train.shape[1]  # Nombre de variables = 2 × nb attributs

        # Opérateur de mutation : probabilité 1/n_vars (chaque variable mutée en moyenne 1 fois)
        mutation  = PolynomialMutation(probability=1.0/n_vars,
                                       distribution_index=MUTATION_INDEX)
        # Opérateur de croisement SBX
        crossover = SBXCrossover(probability=CROSSOVER_PROB,
                                  distribution_index=CROSSOVER_INDEX)

        # Instanciation de l'algorithme avec le budget courant
        if algo_name == 'NSGA-II':
            algo = NSGAII(problem=problem,
                          population_size=pop_size,
                          offspring_population_size=pop_size,  # Même taille enfants/parents
                          mutation=mutation, crossover=crossover,
                          termination_criterion=StoppingByEvaluations(budget))
        else:
            algo = SPEA2(problem=problem,
                         population_size=pop_size,
                         offspring_population_size=pop_size,
                         mutation=mutation, crossover=crossover,
                         termination_criterion=StoppingByEvaluations(budget))

        algo.run()  # Lance l'algorithme jusqu'à épuisement du budget

        # Extraction du front de Pareto et calcul de l'HV
        nd = get_pareto_front(algo.get_result())
        hv = compute_hv(nd, REF_POINT)
        hvs.append(hv)
        print(f'  budget={budget:6d} -> HV={hv:.4f}')
    return hvs


def find_plateau(budgets, hvs, threshold=0.01):
    """
    Identifie le budget où le gain relatif d'hypervolume passe sous le seuil threshold.
    Un gain relatif < 1% signifie que doubler le budget n'apporte plus d'amélioration notable.
    """
    for i in range(1, len(hvs)):
        if hvs[i-1] == 0:
            continue
        gain = (hvs[i] - hvs[i-1]) / max(hvs[i-1], 1e-9)  # Gain relatif (évite la division par 0)
        if gain < threshold:   # Seuil de 1% par défaut
            return budgets[i]  # Premier budget où le gain devient négligeable
    return budgets[-1]         # Si jamais plateau non atteint, retourne le budget max


# ── Lancement de l'étude de convergence ──────────────────────────────────────
print(f'Etude de convergence : {CONV_ALGO} sur {CONV_DATASET}')
print(f'(pop={CONV_POP}, seed={CONV_SEED}, {len(EVAL_BUDGETS)} budgets)\n')

t0 = time.time()
hvs_conv = run_convergence_study(X_pima_tr, y_pima_tr, CONV_ALGO)
print(f'\nTermine en {time.time()-t0:.1f}s')

# ── Détection automatique du plateau ─────────────────────────────────────────
plateau_retenu = find_plateau(EVAL_BUDGETS, hvs_conv)

# ── Tracé de la courbe de convergence ────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(EVAL_BUDGETS, hvs_conv, marker='o', linewidth=2,
        color='steelblue', label=f'{CONV_ALGO} – {CONV_DATASET}')

# Annotation des valeurs d'HV au-dessus de chaque point
for x, y in zip(EVAL_BUDGETS, hvs_conv):
    ax.annotate(f'{y:.3f}', (x, y), textcoords='offset points',
                xytext=(0, 9), ha='center', fontsize=8, color='steelblue')

# Ligne verticale rouge indiquant le budget retenu (plateau)
ax.axvline(plateau_retenu, color='red', linestyle='--', linewidth=1.5,
           label=f'Plateau retenu : {plateau_retenu} eval.')

ax.set_title(f'Courbe de convergence – {CONV_ALGO} sur {CONV_DATASET}\n'
             f'(pop={CONV_POP}, seed={CONV_SEED})',
             fontsize=12, fontweight='bold')
ax.set_xlabel("Nombre d'evaluations", fontsize=11)
ax.set_ylabel('Hypervolume', fontsize=11)
ax.set_xscale('log')   # Échelle logarithmique pour mieux visualiser les petits budgets
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('convergence_study.png', dpi=150, bbox_inches='tight')  # Sauvegarde de la figure
plt.show()

# ── Recommandation basée sur la détection automatique ────────────────────────
print(f'\n{"="*50}')
print(f'Plateau suggere : {plateau_retenu} evaluations')
print(f'-> Modifie MAX_EVALUATIONS dans la cellule suivante')
print(f'{"="*50}')


### Choix manuel du nombre d'évaluations

Regarde la courbe ci-dessus et **choisis toi-même** le budget qui te convient :
- La ligne rouge indique le plateau détecté automatiquement
- Tu peux prendre une valeur **plus basse** si tu veux aller plus vite
- Tu peux prendre une valeur **plus haute** si tu veux plus de qualité

👇 **Modifie uniquement la ligne `MAX_EVALUATIONS = ...` ci-dessous**

In [ ]:
# =============================================================================
# CELLULE 7 – FIXATION DÉFINITIVE DU BUDGET MAX_EVALUATIONS
# =============================================================================
# Après analyse de la courbe de convergence (cellule précédente), l'utilisateur
# doit choisir manuellement la valeur du budget. Modifier uniquement ce nombre.
# ╔══════════════════════════════════════════════════════╗
# ║  MODIFIE CETTE VALEUR selon la courbe de convergence ║
# ╚══════════════════════════════════════════════════════╝

MAX_EVALUATIONS = 500   # ← Budget retenu : compromis qualité / temps de calcul
# Valeur 500 : plateau observé sur la courbe de convergence NSGA-II/PIMA
# À augmenter si la courbe montre encore une progression à 500 évaluations.

print(f'MAX_EVALUATIONS choisi : {MAX_EVALUATIONS}')
print(f'Ce budget sera utilise pour tous les 20 runs suivants.')


## 5. Fonction d'exécution des algorithmes MO

In [ ]:
# =============================================================================
# CELLULE 8 – FONCTIONS D'EXÉCUTION DES ALGORITHMES MO
# =============================================================================
# Définit deux fonctions :
#   - run_mo_algorithm : lance 1 run d'un algorithme MO (NSGA-II ou SPEA2)
#   - run_experiments  : lance N_RUNS runs pour chaque taille de population

def run_mo_algorithm(algo_name, X_train, y_train, pop_size,
                     max_evaluations=MAX_EVALUATIONS, seed=42):
    """
    Lance NSGA-II ou SPEA2 sur le problème PartialClassifMO.
    Retourne la population finale (liste de toutes les solutions, pas seulement le front).

    Paramètres :
        algo_name       : 'NSGA-II' ou 'SPEA2'
        X_train, y_train: données d'entraînement
        pop_size        : taille de la population
        max_evaluations : budget d'évaluations de la fonction objectif
        seed            : graine pour reproductibilité (différente à chaque run)
    """
    # Fixation des graines Python et NumPy pour reproductibilité
    random.seed(seed)
    np.random.seed(seed)

    problem = PartialClassifMO(X_train, y_train)  # Création du problème MO
    n_vars = 2 * X_train.shape[1]                 # Taille du vecteur de variables (2 par attribut)
    mutation_prob = 1.0 / n_vars                   # Probabilité standard : muter chaque variable 1 fois en moyenne

    # ── Opérateurs génétiques ────────────────────────────────────────────────
    mutation  = PolynomialMutation(probability=mutation_prob, distribution_index=MUTATION_INDEX)
    crossover = SBXCrossover(probability=CROSSOVER_PROB, distribution_index=CROSSOVER_INDEX)
    criterion = StoppingByEvaluations(max_evaluations=max_evaluations)  # Critère d'arrêt

    # ── Instanciation de l'algorithme ────────────────────────────────────────
    if algo_name == 'NSGA-II':
        algo = NSGAII(
            problem=problem,
            population_size=pop_size,             # Taille de la population courante
            offspring_population_size=pop_size,   # Même nombre d'individus générés à chaque génération
            mutation=mutation,
            crossover=crossover,
            termination_criterion=criterion
        )
    elif algo_name == 'SPEA2':
        algo = SPEA2(
            problem=problem,
            population_size=pop_size,
            offspring_population_size=pop_size,
            mutation=mutation,
            crossover=crossover,
            termination_criterion=criterion
        )
    else:
        raise ValueError(f'Algorithme inconnu : {algo_name}')

    algo.run()           # Boucle principale de l'algorithme (génération après génération)
    return algo.get_result()  # Population finale (inclut toutes les solutions, pas que le front)


def run_experiments(algo_name, X_train, y_train, pop_sizes=POP_SIZES, n_runs=N_RUNS):
    """
    Lance n_runs exécutions indépendantes pour chaque taille de population.
    Chaque run utilise une graine différente (seed = 100*run + 42) pour varier l'initialisation.

    Retourne : dict { pop_size: { 'hv': [hv_run1, ..., hv_run20],
                                   'fronts': [[sol_run1, ...], ..., [sol_run20, ...]] } }
    """
    results = {}
    for pop_size in pop_sizes:
        print(f'  {algo_name} | pop={pop_size:3d} | ', end='', flush=True)
        hvs, fronts = [], []  # Accumule HV et fronts de Pareto pour chaque run

        for run in range(n_runs):
            seed = 100 * run + 42     # Graine unique par run (déterministe mais variée)
            solutions = run_mo_algorithm(algo_name, X_train, y_train,
                                          pop_size, seed=seed)
            nd_solutions = get_pareto_front(solutions)   # Extrait le front de Pareto final
            hv = compute_hv(nd_solutions, REF_POINT)     # Calcule l'HV du front
            hvs.append(hv)
            fronts.append(nd_solutions)
            print('.', end='', flush=True)  # Indicateur de progression (1 point par run)

        results[pop_size] = {'hv': hvs, 'fronts': fronts}
        # Affiche la moyenne et l'écart-type de l'HV sur les 20 runs
        print(f'  HV moyen={np.mean(hvs):.4f} ± {np.std(hvs):.4f}')

    return results

print('✓ Fonctions run_mo_algorithm et run_experiments définies')


## 6. Exécution NSGA-II

In [ ]:
# =============================================================================
# CELLULE 9 – EXÉCUTION NSGA-II SUR PIMA
# =============================================================================
# Lance les 20 runs × 3 tailles de population de NSGA-II sur le dataset PIMA.
# Résultat stocké dans nsgaii_pima : dict {pop_size: {'hv': [...], 'fronts': [...]}}
# Temps estimé : 1-5 min selon la machine.

print('='*60)
print('NSGA-II – PIMA')
print('='*60)
t0 = time.time()
nsgaii_pima = run_experiments('NSGA-II', X_pima_tr, y_pima_tr)   # 3 pop × 20 runs sur PIMA train
print(f'\nTemps total PIMA : {time.time()-t0:.1f}s')


In [ ]:
# =============================================================================
# CELLULE 10 – EXÉCUTION NSGA-II SUR YEAST1
# =============================================================================
# Lance les 20 runs × 3 tailles de population de NSGA-II sur le dataset Yeast1.
# Résultat stocké dans nsgaii_yeast.
# Yeast1 est plus grand que PIMA (1484 instances) → peut être plus lent.

print('='*60)
print('NSGA-II – Yeast1')
print('='*60)
t0 = time.time()
nsgaii_yeast = run_experiments('NSGA-II', X_yeast_tr, y_yeast_tr)  # 3 pop × 20 runs sur Yeast1 train
print(f'\nTemps total Yeast1 : {time.time()-t0:.1f}s')


## 7. Exécution SPEA2

In [ ]:
# =============================================================================
# CELLULE 11 – EXÉCUTION SPEA2 SUR PIMA
# =============================================================================
# Lance les 20 runs × 3 tailles de population de SPEA2 sur le dataset PIMA.
# Résultat stocké dans spea2_pima.
# SPEA2 maintient une archive externe → peut être légèrement plus lent que NSGA-II.

print('='*60)
print('SPEA2 – PIMA')
print('='*60)
t0 = time.time()
spea2_pima = run_experiments('SPEA2', X_pima_tr, y_pima_tr)  # 3 pop × 20 runs sur PIMA train
print(f'\nTemps total PIMA : {time.time()-t0:.1f}s')


In [ ]:
# =============================================================================
# CELLULE 12 – EXÉCUTION SPEA2 SUR YEAST1
# =============================================================================
# Lance les 20 runs × 3 tailles de population de SPEA2 sur le dataset Yeast1.
# Résultat stocké dans spea2_yeast.
# Après cette cellule, toutes les expériences principales sont terminées.

print('='*60)
print('SPEA2 – Yeast1')
print('='*60)
t0 = time.time()
spea2_yeast = run_experiments('SPEA2', X_yeast_tr, y_yeast_tr)  # 3 pop × 20 runs sur Yeast1 train
print(f'\nTemps total Yeast1 : {time.time()-t0:.1f}s')


## 8. Boxplots de l'Hypervolume par taille de population

Ces boxplots permettent de comparer visuellement la qualité des fronts Pareto  
obtenus selon la taille de population, pour chaque algorithme et chaque jeu de données.

In [ ]:
# =============================================================================
# CELLULE 13 – BOXPLOTS DE L'HYPERVOLUME PAR TAILLE DE POPULATION
# =============================================================================
# Génère une grille 2×2 de boxplots comparant la distribution de l'HV
# sur les 20 runs pour chaque combinaison (algorithme × dataset × taille de pop).
# Permet d'identifier la taille de population optimale et la stabilité des algorithmes.

def plot_hv_boxplots(results_dict, algo_name, dataset_name, ax, color='steelblue'):
    """
    Trace un boxplot de l'hypervolume pour les 3 tailles de population.

    Un boxplot montre :
      - La médiane (trait noir central)
      - Le premier et troisième quartile (boîte)
      - Les moustaches (valeurs non-aberrantes)
      - Les points outliers (valeurs extrêmes)
    """
    data   = [results_dict[p]['hv'] for p in POP_SIZES]  # Liste de 3 listes de 20 HV
    labels = [str(p) for p in POP_SIZES]                  # Labels : '20', '50', '100'

    bp = ax.boxplot(data, labels=labels, patch_artist=True,
                    medianprops=dict(color='black', linewidth=2))
    # Colorisation des boîtes avec transparence
    for patch in bp['boxes']:
        patch.set_facecolor(color)
        patch.set_alpha(0.7)

    ax.set_title(f'{algo_name} – {dataset_name}', fontsize=11, fontweight='bold')
    ax.set_xlabel('Taille de population', fontsize=10)
    ax.set_ylabel('Hypervolume', fontsize=10)
    ax.grid(True, alpha=0.3)  # Grille légère pour faciliter la lecture

# ── Création de la grille 2×2 ────────────────────────────────────────────────
# Ligne 1 : PIMA  | Colonne 1 : NSGA-II | Colonne 2 : SPEA2
# Ligne 2 : Yeast1| Colonne 1 : NSGA-II | Colonne 2 : SPEA2
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

plot_hv_boxplots(nsgaii_pima,  'NSGA-II', 'PIMA',   axes[0, 0], color='steelblue')
plot_hv_boxplots(spea2_pima,   'SPEA2',   'PIMA',   axes[0, 1], color='darkorange')
plot_hv_boxplots(nsgaii_yeast, 'NSGA-II', 'Yeast1', axes[1, 0], color='steelblue')
plot_hv_boxplots(spea2_yeast,  'SPEA2',   'Yeast1', axes[1, 1], color='darkorange')

plt.suptitle('Hypervolume par taille de population\n(20 runs par configuration)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('boxplots_hv.png', dpi=150, bbox_inches='tight')  # Export en PNG haute résolution
plt.show()
print('✓ Boxplots sauvegardés')


## 9. Sélection du meilleur front Pareto et comparaison avec Scikit-Learn

**Protocole :**
1. Pour chaque algorithme, sélectionner le run avec le **meilleur Hypervolume sur le training**.
2. Superposer les points Scikit-Learn (precision, recall) sur le graphe des fronts Pareto.
3. Choisir la **meilleure solution** de chaque front = celle avec le meilleur F1 sur le training.

In [ ]:
# =============================================================================
# CELLULE 14 – SÉLECTION DU MEILLEUR FRONT PARETO PAR ALGORITHME/DATASET
# =============================================================================
# Pour chaque combinaison (algorithme × dataset), identifie parmi les 20 runs :
#   1. Le run ayant l'hypervolume maximal (meilleur front global)
#   2. La solution du front avec le meilleur F1-score (pour comparaison avec sklearn)

def get_best_run(results_dict):
    """
    Parcourt tous les runs et toutes les tailles de population pour trouver
    le front de Pareto avec l'hypervolume le plus élevé.

    Retourne : (pop_size_optimale, front_optimal, hv_maximal)
    """
    best_hv    = -1
    best_pop   = None
    best_front = None
    for pop_size, res in results_dict.items():
        idx = int(np.argmax(res['hv']))          # Index du run avec le HV max pour cette pop_size
        if res['hv'][idx] > best_hv:
            best_hv    = res['hv'][idx]          # Meilleur HV trouvé jusqu'à présent
            best_pop   = pop_size                # Taille de population correspondante
            best_front = res['fronts'][idx]      # Front de Pareto correspondant
    return best_pop, best_front, best_hv


def front_to_prec_rec(front):
    """
    Convertit les objectifs du front (stockés en valeurs négatives pour minimisation)
    en valeurs positives de Precision et Recall pour l'affichage.

    Rappel : objectives[0] = -Precision, objectives[1] = -Recall
    """
    prec = [-s.objectives[0] for s in front]  # -(-Precision) = Precision ∈ [0,1]
    rec  = [-s.objectives[1] for s in front]  # -(-Recall)    = Recall    ∈ [0,1]
    return prec, rec


def best_f1_solution(front, X_train, y_train):
    """
    Parmi toutes les solutions du front de Pareto, sélectionne celle qui
    maximise le F1-score sur le jeu d'entraînement.
    Cette solution est utilisée comme représentant unique du front pour
    la comparaison ponctuelle avec les baselines Scikit-Learn.
    """
    best_f1, best_sol = -1, None
    for s in front:
        yp = predict(s.variables, X_train)              # Prédictions de la règle sur X_train
        f1 = f1_score(y_train, yp, zero_division=0)    # F1-score sur l'ensemble d'entraînement
        if f1 > best_f1:
            best_f1  = f1
            best_sol = s
    return best_sol, best_f1


# ── Sélection pour chaque algorithme et dataset ───────────────────────────────
best = {}  # Dict final : best[dataset][algo] = {'pop', 'front', 'hv', 'best_sol', 'best_f1_train'}

for ds_name, nsgaii_res, spea2_res, X_tr, y_tr in [
        ('PIMA',   nsgaii_pima,  spea2_pima,  X_pima_tr,  y_pima_tr),
        ('Yeast1', nsgaii_yeast, spea2_yeast, X_yeast_tr, y_yeast_tr)]:

    # Meilleur run NSGA-II et SPEA2 pour ce dataset
    n_pop, n_front, n_hv = get_best_run(nsgaii_res)
    s_pop, s_front, s_hv = get_best_run(spea2_res)

    # Meilleure solution (F1 max) dans chaque front
    n_best_sol, n_best_f1 = best_f1_solution(n_front, X_tr, y_tr)
    s_best_sol, s_best_f1 = best_f1_solution(s_front, X_tr, y_tr)

    best[ds_name] = {
        'NSGA-II': {'pop': n_pop, 'front': n_front, 'hv': n_hv,
                    'best_sol': n_best_sol, 'best_f1_train': n_best_f1},
        'SPEA2':   {'pop': s_pop, 'front': s_front, 'hv': s_hv,
                    'best_sol': s_best_sol, 'best_f1_train': s_best_f1},
    }
    print(f'[{ds_name}] NSGA-II → pop={n_pop}, HV={n_hv:.4f}, best_F1_train={n_best_f1:.3f}')
    print(f'[{ds_name}] SPEA2   → pop={s_pop}, HV={s_hv:.4f}, best_F1_train={s_best_f1:.3f}')
    print()


### 9.1 Visualisation des fronts Pareto

In [ ]:
# =============================================================================
# CELLULE 15 – VISUALISATION DES FRONTS PARETO (ENSEMBLE D'ENTRAÎNEMENT)
# =============================================================================
# Trace les fronts de Pareto de NSGA-II et SPEA2 dans l'espace Precision/Recall,
# superposés aux points des baselines Scikit-Learn (RF, SVM, C4.5).
# Chaque étoile marque la solution avec le meilleur F1 dans son front.

fig, axes = plt.subplots(1, 2, figsize=(15, 6))  # 2 sous-graphes côte à côte (PIMA | Yeast1)

# Palettes de couleurs pour les algorithmes et les baselines
colors_algo = {'NSGA-II': 'steelblue', 'SPEA2': 'darkorange'}
markers_sk  = {'RF': 'D', 'SVM': 's', 'C4.5': '^'}    # Formes : losange, carré, triangle
colors_sk   = {'RF': 'green', 'SVM': 'red', 'C4.5': 'purple'}

for ax, ds_name, X_tr, y_tr in [
        (axes[0], 'PIMA',   X_pima_tr,  y_pima_tr),
        (axes[1], 'Yeast1', X_yeast_tr, y_yeast_tr)]:

    # ── Tracé des fronts Pareto des algorithmes évolutionnaires ──────────────
    for algo_name in ['NSGA-II', 'SPEA2']:
        front = best[ds_name][algo_name]['front']
        prec, rec = front_to_prec_rec(front)  # Conversion des objectifs (- → +)

        # Chaque solution du front est un point (recall, precision) dans le graphe
        ax.scatter(rec, prec, color=colors_algo[algo_name], alpha=0.75, s=50,
                   label=f'{algo_name} (pop={best[ds_name][algo_name]["pop"]})')

        # Marquage de la meilleure solution (F1 max) avec une étoile
        sol = best[ds_name][algo_name]['best_sol']
        bp = -sol.objectives[0]   # Precision de la meilleure solution
        br = -sol.objectives[1]   # Recall de la meilleure solution
        ax.scatter(br, bp, color=colors_algo[algo_name], s=200, marker='*',
                   edgecolors='black', linewidths=1.5,
                   label=f'{algo_name} best F1')

    # ── Tracé des baselines Scikit-Learn ─────────────────────────────────────
    for sk_name, res in sklearn_results[ds_name].items():
        # Position = (recall moyen CV, precision moyenne CV)
        ax.scatter(res['recall'], res['precision'],
                   marker=markers_sk[sk_name], color=colors_sk[sk_name],
                   s=150, zorder=5,  # zorder=5 : au premier plan
                   label=f'{sk_name} (F1={res["f1"]:.3f})')
        # Annotation textuelle du nom de l'algorithme
        ax.annotate(sk_name,
                    (res['recall'], res['precision']),
                    textcoords='offset points', xytext=(6, 4), fontsize=9)

    ax.set_xlabel('Sensibilité (Recall)', fontsize=11)
    ax.set_ylabel('Confiance (Precision)', fontsize=11)
    ax.set_title(f'Fronts Pareto + Scikit-Learn – {ds_name}\n(training set)',
                 fontsize=12, fontweight='bold')
    ax.set_xlim(-0.02, 1.05)  # Axes bornés à [0, 1] avec légère marge
    ax.set_ylim(-0.02, 1.05)
    ax.legend(fontsize=8, loc='upper right')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('pareto_fronts_sklearn.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Fronts Pareto sauvegardés')


## 10. Évaluation sur le jeu de test

On évalue les meilleures solutions sélectionnées à l'étape précédente  
(meilleur F1 sur le training) sur le **jeu de test** pour mesurer la généralisation.

In [ ]:
# =============================================================================
# CELLULE 16 – ÉVALUATION DES MEILLEURES SOLUTIONS SUR LE JEU DE TEST
# =============================================================================
# Évalue la meilleure solution de chaque front (F1 max sur train) sur le jeu de TEST
# pour mesurer la généralisation.
# Affiche aussi la règle découverte sous forme lisible.

test_results = {}  # Dict : test_results[dataset][algo] = métriques sur test

for ds_name, X_tr, y_tr, X_te, y_te, feat in [
        ('PIMA',   X_pima_tr,  y_pima_tr,  X_pima_te,  y_pima_te,  features_pima),
        ('Yeast1', X_yeast_tr, y_yeast_tr, X_yeast_te, y_yeast_te, features_yeast)]:

    print(f'\n{"="*55}')
    print(f'Dataset : {ds_name}')
    print(f'{"="*55}')
    test_results[ds_name] = {}

    for algo_name in ['NSGA-II', 'SPEA2']:
        sol = best[ds_name][algo_name]['best_sol']  # Solution sélectionnée (F1 max sur train)

        # ── Évaluation sur l'ensemble d'ENTRAÎNEMENT (pour vérification) ─────
        yp_tr = predict(sol.variables, X_tr)                            # Prédictions sur train
        f1_tr  = f1_score(y_tr, yp_tr, zero_division=0)
        pr_tr  = precision_score(y_tr, yp_tr, zero_division=0)
        rc_tr  = recall_score(y_tr, yp_tr, zero_division=0)

        # ── Évaluation sur l'ensemble de TEST (généralisation) ───────────────
        yp_te = predict(sol.variables, X_te)                            # Prédictions sur test
        f1_te  = f1_score(y_te, yp_te, zero_division=0)
        pr_te  = precision_score(y_te, yp_te, zero_division=0)
        rc_te  = recall_score(y_te, yp_te, zero_division=0)
        acc_te = accuracy_score(y_te, yp_te)  # Précision globale (toutes classes confondues)

        # Stockage des résultats pour le tableau récapitulatif
        test_results[ds_name][algo_name] = {
            'precision_train': pr_tr, 'recall_train': rc_tr, 'f1_train': f1_tr,
            'precision_test':  pr_te, 'recall_test':  rc_te, 'f1_test':  f1_te,
            'accuracy_test':   acc_te,
            'hv': best[ds_name][algo_name]['hv'],   # Hypervolume du meilleur run
            'pop': best[ds_name][algo_name]['pop'],  # Taille de population optimale
        }

        print(f'\n  ── {algo_name} (pop={best[ds_name][algo_name]["pop"]}) ──')
        print(f'    Train  | Prec={pr_tr:.3f} | Rec={rc_tr:.3f} | F1={f1_tr:.3f}')
        print(f'    Test   | Prec={pr_te:.3f} | Rec={rc_te:.3f} | F1={f1_te:.3f} | Acc={acc_te:.3f}')
        print(f'    HV (best run) = {best[ds_name][algo_name]["hv"]:.4f}')
        print(f'    Règle découverte :')
        decode_rule(sol.variables, feat)  # Affiche la règle SI...ALORS lisible


## 11. Tableau comparatif final

In [ ]:
# =============================================================================
# CELLULE 17 – TABLEAU RÉCAPITULATIF COMPARATIF (TEST SET)
# =============================================================================
# Construit un DataFrame pandas comparant tous les algorithmes (Scikit-Learn + MO)
# sur le jeu de TEST pour les deux datasets.
# Colonnes : Algorithme | Confiance (Precision) | Sensibilité (Recall) | F1 | HV | Type

def fmt(v): return f'{v:.3f}'  # Formate un flottant avec 3 décimales

rows = []  # Liste de dicts, chaque dict = une ligne du tableau

for ds_name, X_te, y_te in [('PIMA', X_pima_te, y_pima_te),
                              ('Yeast1', X_yeast_te, y_yeast_te)]:
    sk = sklearn_results[ds_name]   # Résultats Scikit-Learn (CV)
    mo = test_results[ds_name]      # Résultats MO (test set)

    # ── Lignes pour les classifieurs Scikit-Learn ──────────────────────────
    for sk_name, fn in [('RF', rf_fn), ('SVM', svm_fn), ('C4.5', c45_fn)]:
        model = fn(42)  # Recréation du modèle avec seed fixe
        # Re-entraîne sur X_train et prédit sur X_test (évaluation cohérente avec les AG MO)
        if ds_name == 'PIMA':
            model.fit(X_pima_tr, y_pima_tr); yp = model.predict(X_pima_te)
        else:
            model.fit(X_yeast_tr, y_yeast_tr); yp = model.predict(X_yeast_te)

        rows.append({
            'Jeu':        ds_name,
            'Algorithme': sk_name,
            'Confiance':  fmt(precision_score(y_te, yp, zero_division=0)),
            'Sensibilité':fmt(recall_score(y_te, yp, zero_division=0)),
            'F1-mesure':  fmt(f1_score(y_te, yp, zero_division=0)),
            'HV':         '—',  # Pas d'hypervolume pour les classifieurs classiques
            'Type':       'Boîte noire' if sk_name != 'C4.5' else 'Boîte blanche'
            # RF et SVM : boîtes noires (décision non interprétable directement)
            # C4.5 : boîte blanche (règles lisibles via l'arbre)
        })

    # ── Lignes pour les algorithmes évolutionnaires MO ────────────────────
    for algo_name in ['NSGA-II', 'SPEA2']:
        r = mo[algo_name]
        rows.append({
            'Jeu':        ds_name,
            'Algorithme': algo_name,
            'Confiance':  fmt(r['precision_test']),
            'Sensibilité':fmt(r['recall_test']),
            'F1-mesure':  fmt(r['f1_test']),
            'HV':         f'{r["hv"]:.4f}',  # Hypervolume disponible pour les AG MO
            'Type':       'Boîte blanche (règle)'  # Règle SI...ALORS directement lisible
        })

# ── Affichage du tableau ─────────────────────────────────────────────────────
df_final = pd.DataFrame(rows)

for ds_name in ['PIMA', 'Yeast1']:
    print(f'\n{"="*70}')
    print(f'  Tableau récapitulatif – {ds_name}  (évaluation sur le TEST)')
    print(f'{"="*70}')
    sub = df_final[df_final['Jeu'] == ds_name].drop(columns='Jeu')
    print(sub.to_string(index=False))


### 11.1 Graphe comparatif Precision–Recall sur le test

In [ ]:
# =============================================================================
# CELLULE 18 – GRAPHE COMPARATIF PRECISION/RECALL SUR LE JEU DE TEST
# =============================================================================
# Reproduit le graphe Precision/Recall mais évalué sur le TEST (et non le train).
# Permet de vérifier la généralisation : les règles MO restent-elles compétitives ?
# Points = solutions des fronts Pareto évaluées sur X_te ; étoiles = meilleure solution F1.

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

for ax, ds_name, X_te, y_te in [
        (axes[0], 'PIMA',   X_pima_te,  y_pima_te),
        (axes[1], 'Yeast1', X_yeast_te, y_yeast_te)]:

    # ── Front de Pareto évalué sur le TEST ───────────────────────────────────
    for algo_name in ['NSGA-II', 'SPEA2']:
        front = best[ds_name][algo_name]['front']
        precs, recs = [], []
        for s in front:
            yp = predict(s.variables, X_te)    # Applique la règle encodée sur X_te
            precs.append(precision_score(y_te, yp, zero_division=0))
            recs.append(recall_score(y_te, yp, zero_division=0))

        ax.scatter(recs, precs, color=colors_algo[algo_name], alpha=0.6, s=40,
                   label=f'{algo_name} (front test)')

        # Étoile = meilleure solution (F1 max sur train) évaluée sur le test
        sol = best[ds_name][algo_name]['best_sol']
        yp = predict(sol.variables, X_te)
        bp = precision_score(y_te, yp, zero_division=0)  # Precision sur test
        br = recall_score(y_te, yp, zero_division=0)     # Recall sur test
        ax.scatter(br, bp, color=colors_algo[algo_name], s=200, marker='*',
                   edgecolors='black', linewidths=1.5)

    # ── Baselines Scikit-Learn évaluées sur le TEST ───────────────────────
    for sk_name, fn in [('RF', rf_fn), ('SVM', svm_fn), ('C4.5', c45_fn)]:
        model = fn(42)
        # Entraînement sur X_train, prédiction sur X_test
        if ds_name == 'PIMA':
            model.fit(X_pima_tr, y_pima_tr); yp = model.predict(X_pima_te)
        else:
            model.fit(X_yeast_tr, y_yeast_tr); yp = model.predict(X_yeast_te)
        pr = precision_score(y_te, yp, zero_division=0)
        rc = recall_score(y_te, yp, zero_division=0)
        ax.scatter(rc, pr, marker=markers_sk[sk_name], color=colors_sk[sk_name],
                   s=150, zorder=5, label=f'{sk_name}')
        ax.annotate(sk_name, (rc, pr), textcoords='offset points', xytext=(6, 4), fontsize=9)

    ax.set_xlabel('Sensibilité (Recall)', fontsize=11)
    ax.set_ylabel('Confiance (Precision)', fontsize=11)
    ax.set_title(f'Précision–Rappel sur le TEST – {ds_name}', fontsize=12, fontweight='bold')
    ax.set_xlim(-0.02, 1.05)
    ax.set_ylim(-0.02, 1.05)
    ax.legend(fontsize=8, loc='upper right')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('comparison_test.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Graphe comparatif test sauvegardé')


In [ ]:
# =============================================================================
# CELLULE 19 – GRAPHES DE CONVERGENCE DE L'HYPERVOLUME
# =============================================================================
# Trace l'évolution de l'hypervolume moyen (± écart-type) en fonction du nombre
# d'évaluations pour NSGA-II et SPEA2 sur les deux datasets.
# Requiert que conv_results soit défini (résultats de l'étude de convergence étendue).

# ── Tracé des graphes de convergence ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
colors = {'NSGA-II': 'steelblue', 'SPEA2': 'darkorange'}  # Couleurs cohérentes avec les autres graphes

for ax, ds_name in zip(axes, ['PIMA', 'Yeast1']):
    for algo in ['NSGA-II', 'SPEA2']:
        key  = f'{algo}_{ds_name}'   # Clé d'accès dans conv_results (ex: 'NSGA-II_PIMA')
        data = conv_results[key]
        evals_list = data['evals']   # Liste de listes : nb d'évaluations à chaque snapshot
        hvs_list   = data['hvs']     # Liste de listes : HV à chaque snapshot pour chaque run

        # ── Alignement des séries sur une grille commune ─────────────────────
        # Certains runs peuvent avoir des longueurs légèrement différentes → on prend le minimum
        min_len   = min(len(e) for e in evals_list)
        evals_arr = evals_list[0][:min_len]              # Axe X : nombre d'évaluations
        hvs_mat   = np.array([h[:min_len] for h in hvs_list])  # Matrice (n_runs × n_points)

        mean_hv = hvs_mat.mean(axis=0)   # Moyenne sur les runs (axe 0 = dimension runs)
        std_hv  = hvs_mat.std(axis=0)    # Écart-type sur les runs

        # ── Courbe de convergence moyenne ────────────────────────────────────
        ax.plot(evals_arr, mean_hv,
                color=colors[algo], lw=2,
                label=f"{algo} (pop={data['pop']})")

        # ── Bande de confiance ± 1 écart-type ────────────────────────────────
        # Visualise la variabilité inter-runs (robustesse de l'algorithme)
        ax.fill_between(evals_arr,
                        mean_hv - std_hv,   # Borne inférieure
                        mean_hv + std_hv,   # Borne supérieure
                        color=colors[algo], alpha=0.2)  # Transparence pour superposition lisible

    ax.set_title(f"Convergence de l'Hypervolume – {ds_name}", fontsize=13)
    ax.set_xlabel("Nombre d'évaluations", fontsize=11)
    ax.set_ylabel('Hypervolume', fontsize=11)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)

plt.suptitle('Graphes de convergence : HV moyen ± écart-type (5 runs)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('convergence_hv.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Graphes sauvegardés dans convergence_hv.png')


## 12. Analyse des résultats

### 12.1 Analyse des algorithmes MO

**NSGA-II vs SPEA2 :**
- NSGA-II utilise le tri par rang de dominance et la distance de crowding pour maintenir la diversité.  
- SPEA2 utilise une archive externe de solutions non-dominées et une estimation de densité basée sur les k-ièmes voisins.  
- En général, SPEA2 produit des fronts mieux répartis, mais NSGA-II est souvent plus rapide à converger.

**Impact de la taille de population :**
- Une petite population (20) converge rapidement mais manque de diversité → front Pareto pauvre.  
- Une grande population (100) explore mieux l'espace mais requiert plus d'évaluations.  
- La taille optimale se lit sur les boxplots : c'est la plus petite taille donnant un HV stable.

### 12.2 Analyse de l'interprétabilité

Les algorithmes génétiques (AG MO) produisent des **règles lisibles** :  
> *SI Glucose ∈ [120, 180] ET BMI ∈ [28, 45] → classe positive (diabétique)*

Avantages par rapport aux boîtes noires (RF, SVM) :
1. **Explicabilité** : le décideur comprend les critères de décision.
2. **Adaptabilité** : on peut choisir une solution avec plus de confiance ou plus de sensibilité selon le besoin médical.
3. **Compromis explicite** : le front Pareto montre tous les compromis disponibles.

Limitation : une seule règle Michigan couvre un sous-espace limité ; la représentation Pittsburgh (plusieurs règles) permettrait une meilleure couverture.

### 12.3 Comparaison avec Scikit-Learn

| Aspect | AG MO (Michigan) | Random Forest | SVM | C4.5 |
|--------|-----------------|---------------|-----|------|
| Interprétabilité | ✅ Règle lisible | ❌ Boîte noire | ❌ Boîte noire | ⚠️ Arbre |
| Confiance | Variable | Élevée | Très élevée | Moyenne |
| Sensibilité | Variable | Modérée | Faible | Modérée |
| Compromis explicite | ✅ Front Pareto | ❌ | ❌ | ❌ |
| Temps calcul | Lent | Rapide | Rapide | Rapide |

Les AG MO offrent une **valeur ajoutée** en termes d'interprétabilité et de flexibilité des compromis,  
au prix d'un temps de calcul plus élevé et de performances légèrement inférieures en F1.

## 13. Conclusions

### Résultats principaux

1. **Qualité des fronts Pareto** : NSGA-II et SPEA2 produisent tous deux des fronts Pareto couvrant un large spectre de compromis confiance/sensibilité, démontrant la valeur de l'approche multi-objectif.

2. **Paramètre population** : L'Hypervolume augmente significativement entre pop=20 et pop=50, puis se stabilise à pop=100. Le gain marginal de pop=100 par rapport à pop=50 est modeste — la taille optimale semble être **50 pour PIMA** et **100 pour Yeast1** (plus déséquilibré).

3. **Comparaison avec Scikit-Learn** :
   - Les algorithmes Scikit-Learn restent supérieurs en F1-mesure sur le test.
   - Cependant, les AG MO offrent un **front de solutions** permettant d'ajuster le compromis precision/recall selon le contexte métier.
   - La règle produite est directement **exploitable** par un décideur non-expert.

4. **Interprétabilité** : Les règles découvertes sont concises (2 attributs actifs en moyenne), ce qui les rend facilement communicables à des médecins ou décideurs.

### Perspectives

- Utiliser la **représentation Pittsburgh** (plusieurs règles) pour améliorer la couverture et le recall.
- Tester d'autres algorithmes MO : MOEA/D (décomposition en sous-problèmes scalaires) ou SMS-EMOA.
- Augmenter le nombre d'évaluations ou utiliser un critère d'arrêt adaptatif.
- Appliquer la sélection de variables pour réduire la dimensionnalité et améliorer l'interprétabilité.